# Vectorized micrograd — FER2013 Demo

Seven-class facial expression recognition on the FER2013 dataset —
the standard public benchmark for the same task studied in the NICE paper
(Dinh et al., ICLR 2015) under the Toronto Face Database (TFD).

| | MNIST | FER2013 |
|---|---|---|
| Input | 28 × 28 = 784 dims | 48 × 48 = 2 304 dims |
| Classes | 10 digits | 7 expressions |
| Train set | 60 000 | 28 709 |
| Test set | 10 000 | 3 589 |
| Human accuracy | ~98 % | ~65 % |

FER2013 is a genuinely hard dataset — human labellers agree only ~65 % of
the time, and the class distribution is heavily skewed toward *Happy*.

**Prerequisites — download the dataset before running this notebook.**

1. Create a free Kaggle account at <https://www.kaggle.com>.
2. Go to <https://www.kaggle.com/datasets/msambare/fer2013> and click **Download**.
3. Extract the zip. You will find two folders: `train/` and `test/`.  
   Each contains seven subfolders named after the expression class.
4. Place both folders in the same directory as this notebook:
   ```
   fer2013_demo.ipynb
   train/
     angry/   disgust/   fear/   happy/   neutral/   sad/   surprise/
   test/
     angry/   disgust/   fear/   happy/   neutral/   sad/   surprise/
   ```

The download is ~60 MB. Everything else runs with no extra dependencies beyond
Pillow (PIL), which ships with most Python distributions.

## Engine at a glance

`vect_engine.py` keeps the same interface as Karpathy's scalar `Value`
— `backward()`, `relu()`, operator overloads — but every node stores a full
NumPy array. The two pieces that make this non-trivial are:

* **`_unbroadcast`** — sums a broadcasted gradient back to the operand shape.
  When `b` has shape `(64,)` and `out = X @ W + b` has shape `(256, 64)`,
  the bias gradient must be summed over the batch dimension.
* **`__matmul__`** — explicit 2D backward
  (`dL/dX = dL/dY @ W.T`, `dL/dW = X.T @ dL/dY`).

With those two pieces in place the rest of the engine — add, mul, relu,
tanh, softmax_ce — is identical in structure to the scalar version.

## Building the MLP

`vect_nn.py` has two classes:

* **`Layer`** — weight matrix `W` (nin × nout) and bias `b` (nout,).  
  He initialisation: `W ~ N(0, sqrt(2/nin))`.
* **`MLP`** — stacks layers, applies the chosen activation between hidden
  layers, leaves the last layer linear.

FER2013 inputs are 2 304-dimensional (48 × 48 pixels flattened) — three
times wider than MNIST.  We use a `2304 → 512 → 256 → 7` network; the
wider first layer compensates for the richer input space.

In [ ]:
import csv
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from vect_micrograd.vect_engine import Value
from vect_micrograd.vect_nn import MLP
from vect_micrograd.optim import Adam
from vect_micrograd.utils import one_hot, cross_entropy_loss, save_checkpoint, load_checkpoint

EMOTION_NAMES = ['Angry', 'Disgust', 'Fear', 'Happy', 'Sad', 'Surprise', 'Neutral']
N_CLASSES     = len(EMOTION_NAMES)

## Loading FER2013

The Kaggle dataset uses an **image-folder layout**: one subfolder per class
inside `train/` and `test/`. The subfolder name is the label.

```
train/angry/   *.jpg   ← 3 995 images
train/happy/   *.jpg   ← 7 215 images
...
test/angry/    *.jpg   ←   467 images
...
```

The loader below reads every JPEG with `PIL.Image`, converts to greyscale,
flattens the 48 × 48 pixels to a 2 304-element vector, and normalises to
`[0, 1]`.  Class indices follow alphabetical subfolder order so they are
fully determined by the directory — no external label file needed.

In [ ]:
import os
from PIL import Image

DATA_ROOT = '.'   # directory that contains train/ and test/

def load_fer2013_folders(root):
    """Load FER2013 from the image-folder layout used on Kaggle.

    Expected tree
    -------------
    root/
      train/
        angry/  disgust/  fear/  happy/  neutral/  sad/  surprise/
      test/
        angry/  disgust/  fear/  happy/  neutral/  sad/  surprise/

    Each leaf folder contains JPEG images (48 × 48 greyscale).

    Returns
    -------
    X_train : (28709, 2304) float64 in [0, 1]
    y_train : (28709,)      int
    X_test  : ( 3589, 2304) float64 in [0, 1]
    y_test  : ( 3589,)      int
    classes : list of str, length 7 (alphabetical order)
    """
    train_dir = os.path.join(root, 'train')
    classes   = sorted(os.listdir(train_dir))   # alphabetical → reproducible
    label_map = {cls: i for i, cls in enumerate(classes)}

    def read_split(split):
        X, y = [], []
        for cls in classes:
            folder = os.path.join(root, split, cls)
            if not os.path.isdir(folder):
                continue
            for fname in sorted(os.listdir(folder)):
                if not fname.lower().endswith(('.jpg', '.jpeg', '.png')):
                    continue
                img = Image.open(os.path.join(folder, fname)).convert('L')
                arr = np.array(img, dtype=float).ravel() / 255.0
                X.append(arr)
                y.append(label_map[cls])
        return np.stack(X), np.array(y, dtype=int)

    print('  reading train/ ...')
    X_train, y_train = read_split('train')
    print('  reading test/ ...')
    X_test,  y_test  = read_split('test')
    return X_train, y_train, X_test, y_test, classes


print(f'Reading from {os.path.abspath(DATA_ROOT)} ...')
X_train_raw, y_train_raw, X_test, y_test, EMOTION_NAMES = \
    load_fer2013_folders(DATA_ROOT)
N_CLASSES = len(EMOTION_NAMES)

print(f'Train : {X_train_raw.shape}  |  Test : {X_test.shape}')
print(f'Classes ({N_CLASSES}): {EMOTION_NAMES}')
print(f'Pixel range: [{X_train_raw.min():.2f}, {X_train_raw.max():.2f}]')

## Class distribution

FER2013 is **not** balanced — *Happy* accounts for ~25 % of training examples
while *Disgust* is less than 2 %.  This imbalance is one of the reasons the
dataset is hard: a model that ignores Disgust still scores well on raw accuracy.

Rather than resampling, we draw a fixed quota per class so the MLP sees each
expression equally during training.

In [ ]:
# --- raw class counts ---
counts_raw = np.bincount(y_train_raw, minlength=N_CLASSES)

fig, ax = plt.subplots(figsize=(8, 3))
bars = ax.bar(EMOTION_NAMES, counts_raw, color='steelblue', edgecolor='white')
ax.set_ylabel('count')
ax.set_title('FER2013 training set — raw class distribution')
for bar, v in zip(bars, counts_raw):
    ax.text(bar.get_x() + bar.get_width() / 2, v + 30,
            str(v), ha='center', va='bottom', fontsize=8)
plt.tight_layout()
plt.show()

print('Counts per class:')
for name, cnt in zip(EMOTION_NAMES, counts_raw):
    print(f'  {name:<10s}: {cnt:5d}  ({cnt/len(y_train_raw):.1%})')

## Balanced training subset

We cap each class at `N_PER_CLASS` samples. *Disgust* has only ~436 training
images, so that sets the natural ceiling.  Using 400 per class gives
2 800 training examples — comparable to the 5 000 used in the MNIST demo
relative to dataset size.

The full 3 589-sample test set is kept intact.

In [ ]:
N_PER_CLASS = 400
rng = np.random.default_rng(0)

idx = np.concatenate([
    rng.choice(np.where(y_train_raw == c)[0], N_PER_CLASS, replace=False)
    for c in range(N_CLASSES)
])
rng.shuffle(idx)

X_train, y_train = X_train_raw[idx], y_train_raw[idx]
Y_train = one_hot(y_train, classes=N_CLASSES)

print(f'Training subset : {X_train.shape}  (balanced, {N_PER_CLASS} per class)')
print(f'Test set        : {X_test.shape}  (full PrivateTest split)')

counts_sub = np.bincount(y_train, minlength=N_CLASSES)
fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(EMOTION_NAMES, counts_sub, color='steelblue', edgecolor='white')
ax.set_ylabel('count')
ax.set_title(f'Training subset — {N_PER_CLASS} per class')
plt.tight_layout()
plt.show()

## A look at the data

Each image is a 48 × 48 greyscale face aligned and cropped to a fixed bounding
box.  Unlike MNIST the intra-class variation is high — the same expression can
look very different across age, gender, lighting, and ethnicity.

In [ ]:
fig, axes = plt.subplots(2, N_CLASSES, figsize=(13, 4))

for c, name in enumerate(EMOTION_NAMES):
    examples = X_train[y_train == c][:2]
    for row, img in enumerate(examples):
        axes[row, c].imshow(img.reshape(48, 48), cmap='gray', vmin=0, vmax=1)
        axes[row, c].axis('off')
        if row == 0:
            axes[row, c].set_title(name, fontsize=9)

plt.suptitle('Two examples per expression class', y=1.02)
plt.tight_layout()
plt.show()

## Model

A three-layer MLP with ReLU activations:

```
2304  →  512  →  256  →  7
         ReLU    ReLU    (linear logits)
```

The first hidden layer is wider than in the MNIST demo to handle the richer
2 304-dimensional input space. The final layer produces raw logits; `softmax_ce`
turns them into probabilities and computes cross-entropy loss in one
numerically stable fused operation.

In [ ]:
np.random.seed(42)

model = MLP(2304, [512, 256, 7])

print(model)
print(f'Scalar parameters : {sum(p.data.size for p in model.parameters()):,}')
print(f'Value nodes       : {len(model.parameters())}  '
      f'(vs ~{sum(p.data.size for p in model.parameters()):,} in scalar micrograd)')

## Training

Mini-batch Adam with a linear learning-rate decay to 10 % of the initial value.
The training loop mirrors the MNIST demo:

1. Sample a random mini-batch.
2. Forward pass → loss.
3. Zero gradients, backward, optimizer step.
4. Track the best checkpoint (lowest batch loss).

Every 50 steps the full training-set loss and accuracy are reported.

> **Note on expected accuracy.** Human accuracy on FER2013 is ~65 %. A plain
> MLP trained on 2 800 balanced examples should reach roughly 40–50 % test
> accuracy — well above the 14 % random baseline, and in the same ballpark as
> SVMs trained on raw pixels.

In [ ]:
STEPS      = 500
BATCH_SIZE = 256
L2_ALPHA   = 1e-4

optimizer = Adam(model.parameters(), lr=1e-3, total_steps=STEPS)

best_checkpoint, best_loss, best_step = None, float('inf'), -1
history = []   # (step, batch_loss, batch_acc, full_loss, full_acc)

rng = np.random.default_rng(7)

for k in range(STEPS):
    ri = rng.integers(0, len(X_train), BATCH_SIZE)
    Xb = X_train[ri]
    Yb = Y_train[ri]

    batch_loss, batch_acc = cross_entropy_loss(model, Xb, Yb, alpha=L2_ALPHA)

    if batch_loss.item() < best_loss:
        best_loss = batch_loss.item()
        best_step = k
        best_checkpoint = save_checkpoint(model)

    optimizer.zero_grad()
    batch_loss.backward()
    optimizer.step(k)

    if k % 50 == 0:
        full_loss, full_acc = cross_entropy_loss(model, X_train, Y_train)
        history.append((k, batch_loss.item(), batch_acc, full_loss.item(), full_acc))
        print(f'step {k:3d}  '
              f'batch loss {batch_loss.item():.4f}  batch acc {batch_acc:.2%}  '
              f'full acc {full_acc:.2%}')

print(f'\nBest checkpoint at step {best_step}, batch loss {best_loss:.4f}')

## Training curves

In [ ]:
hist         = np.array(history)
steps_logged = hist[:, 0]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(steps_logged, hist[:, 1], color='steelblue',   label='batch loss')
ax1.plot(steps_logged, hist[:, 3], color='navy',        label='full loss', linestyle='--')
ax1.set_xlabel('step')
ax1.set_ylabel('cross-entropy')
ax1.set_title('Loss')
ax1.legend()

ax2.plot(steps_logged, hist[:, 2], color='darkorange',  label='batch acc')
ax2.plot(steps_logged, hist[:, 4], color='saddlebrown', label='full acc',  linestyle='--')
ax2.axhline(1/N_CLASSES, color='gray', linestyle=':', label=f'chance ({1/N_CLASSES:.0%})')
ax2.set_xlabel('step')
ax2.set_ylabel('accuracy')
ax2.set_ylim(0, 1)
ax2.set_title('Accuracy')
ax2.legend()

plt.suptitle('FER2013 training (2 800 samples, mini-batch Adam)')
plt.tight_layout()
plt.show()

## Test-set evaluation

Restore the best checkpoint and run a single forward pass over the full
3 589-sample PrivateTest split.

In [ ]:
load_checkpoint(model, best_checkpoint)

test_logits = model(Value(X_test))
test_preds  = test_logits.data.argmax(axis=1)
test_acc    = float((test_preds == y_test).mean())

print(f'Restored checkpoint from step {best_step}')
print(f'Test accuracy : {test_acc:.2%}  '
      f'({int(test_acc * len(y_test)):,} / {len(y_test):,} correct)')
print(f'Chance baseline : {1/N_CLASSES:.2%}')
print(f'Human accuracy  : ~65 %')

## Confusion matrix

Which expressions does the model confuse most? The off-diagonal structure
reveals systematic errors — e.g. *Fear* and *Sad* are commonly swapped,
and *Disgust* often collapses into *Angry*.

In [ ]:
C = np.zeros((N_CLASSES, N_CLASSES), dtype=int)
for true, pred in zip(y_test, test_preds):
    C[true, pred] += 1

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(C, cmap='Blues')
plt.colorbar(im, ax=ax, fraction=0.046)

ax.set_xticks(range(N_CLASSES))
ax.set_yticks(range(N_CLASSES))
ax.set_xticklabels(EMOTION_NAMES, rotation=35, ha='right')
ax.set_yticklabels(EMOTION_NAMES)
ax.set_xlabel('predicted')
ax.set_ylabel('true')
ax.set_title('Confusion matrix (PrivateTest set)')

for i in range(N_CLASSES):
    for j in range(N_CLASSES):
        color = 'white' if C[i, j] > C.max() / 2 else 'black'
        ax.text(j, i, str(C[i, j]), ha='center', va='center', fontsize=9, color=color)

plt.tight_layout()
plt.show()

## Hard cases — misclassified faces

A random sample of mistakes. Note that many of these are ambiguous even
to a human — the dataset label is not always the only reasonable reading
of the expression.

In [ ]:
wrong_idx = np.where(test_preds != y_test)[0]
show      = np.random.default_rng(1).choice(wrong_idx, size=14, replace=False)

fig, axes = plt.subplots(2, 7, figsize=(13, 4))
for ax, i in zip(axes.ravel(), show):
    ax.imshow(X_test[i].reshape(48, 48), cmap='gray', vmin=0, vmax=1)
    t = EMOTION_NAMES[y_test[i]]
    p = EMOTION_NAMES[test_preds[i]]
    ax.set_title(f't: {t}\np: {p}', fontsize=7)
    ax.axis('off')

plt.suptitle('Misclassified examples  (t = true label, p = predicted)', y=1.02)
plt.tight_layout()
plt.show()

print(f'{len(wrong_idx)} misclassifications out of {len(y_test)} test images')

## Per-class accuracy

Class-level accuracy exposes the imbalance effect even with balanced training:
*Happy* is typically the easiest class while *Disgust* and *Fear* are hardest,
consistent with the literature.

In [ ]:
per_class = [
    float((test_preds[y_test == c] == c).mean())
    for c in range(N_CLASSES)
]

# test set size per class (imbalanced)
test_counts = np.bincount(y_test, minlength=N_CLASSES)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

# --- per-class accuracy ---
bars = ax1.bar(EMOTION_NAMES, per_class, color='steelblue', edgecolor='white')
ax1.axhline(test_acc,      color='darkorange', linestyle='--', label=f'overall {test_acc:.2%}')
ax1.axhline(1/N_CLASSES,   color='gray',       linestyle=':',  label=f'chance {1/N_CLASSES:.0%}')
ax1.set_ylim(0, 1)
ax1.set_ylabel('accuracy')
ax1.set_title('Per-class test accuracy')
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0%}'))
ax1.tick_params(axis='x', rotation=30)
ax1.legend(fontsize=8)
for bar, v in zip(bars, per_class):
    ax1.text(bar.get_x() + bar.get_width() / 2, v + 0.01,
             f'{v:.0%}', ha='center', va='bottom', fontsize=8)

# --- test set class distribution ---
ax2.bar(EMOTION_NAMES, test_counts, color='slategray', edgecolor='white')
ax2.set_ylabel('count')
ax2.set_title('Test set class distribution')
ax2.tick_params(axis='x', rotation=30)
for bar, v in zip(ax2.patches, test_counts):
    ax2.text(bar.get_x() + bar.get_width() / 2, v + 5,
             str(v), ha='center', va='bottom', fontsize=8)

plt.suptitle('FER2013 — test set diagnostics')
plt.tight_layout()
plt.show()

## Softmax probability distributions

For a handful of correct and incorrect predictions, we plot the full softmax
distribution to see how confident the model is. A peaked distribution on the
wrong class is a harder failure than a flat, uncertain one.

In [ ]:
# Compute softmax probabilities for the full test set
logits_np = test_logits.data                          # (3589, 7)
logits_np -= logits_np.max(axis=1, keepdims=True)     # numerical stability
exp_l      = np.exp(logits_np)
probs_all  = exp_l / exp_l.sum(axis=1, keepdims=True) # (3589, 7)

correct_idx   = np.where(test_preds == y_test)[0]
incorrect_idx = np.where(test_preds != y_test)[0]

rng_show = np.random.default_rng(3)
show_correct   = rng_show.choice(correct_idx,   size=4, replace=False)
show_incorrect = rng_show.choice(incorrect_idx, size=4, replace=False)
show_all = np.concatenate([show_correct, show_incorrect])

fig, axes = plt.subplots(2, 8, figsize=(15, 5),
                         gridspec_kw={'height_ratios': [2, 1]})

for col, i in enumerate(show_all):
    # face image
    axes[0, col].imshow(X_test[i].reshape(48, 48), cmap='gray', vmin=0, vmax=1)
    axes[0, col].axis('off')
    correct = (test_preds[i] == y_test[i])
    color   = 'green' if correct else 'red'
    axes[0, col].set_title(
        f"{EMOTION_NAMES[y_test[i]]}\n→ {EMOTION_NAMES[test_preds[i]]}",
        fontsize=7, color=color)

    # probability bar
    bar_colors = ['green' if j == y_test[i] else
                  'red'   if j == test_preds[i] else
                  'lightgray'
                  for j in range(N_CLASSES)]
    axes[1, col].bar(range(N_CLASSES), probs_all[i], color=bar_colors, edgecolor='white')
    axes[1, col].set_ylim(0, 1)
    axes[1, col].set_xticks(range(N_CLASSES))
    axes[1, col].set_xticklabels([n[:3] for n in EMOTION_NAMES], fontsize=6, rotation=45)
    axes[1, col].set_yticks([])
    if col == 0:
        axes[1, col].set_ylabel('P', fontsize=8)

axes[0, 0].set_ylabel('Correct ✓', fontsize=9, color='green')
axes[0, 4].set_ylabel('Wrong ✗', fontsize=9, color='red')

plt.suptitle('Softmax confidence  |  green = true class, red = predicted class', y=1.02)
plt.tight_layout()
plt.show()